# 03. 기준 답(정답 단계) · 현장 용어

**목적**
1. 직무 영상을 보고 사람이 정한 "올바른 단계 목록"을 `data/gold/<job_key>.csv` 로 만든다 → AI 결과 평가 기준 (인계서 10장)
2. 전사문에서 현장 용어를 골라 `data/curated/field_terms.csv` 에 넣는다 → 어휘 점검에서 쪼개지지 않게 (5.6)

전사문은 파이프라인이 `data/review/<job_key>/transcript.csv` (start, end, text) 로 만들어 커밋한다.

기준 답 작성 규칙 (AI 추출 규칙과 같게)
- 한 행 = 행동 하나. 화면·음성으로 확인되는 것만.
- tools·cautions 는 여러 개면 `;` 로 구분. 근거 구간은 MM:SS.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent            # notebooks/ 에서 실행한다고 가정
sys.path.insert(0, str(ROOT / "backend"))
DATA = ROOT / "data"
import pandas as pd
pd.set_option("display.max_colwidth", 80)

JOB = "kitchen_helper"
tr = pd.read_csv(DATA / f"review/{JOB}/transcript.csv")
tr

## 1) 현장 용어 후보: 사전에 없어서 쪼개지는 명사

In [ ]:
from kiwipiepy import Kiwi
from collections import Counter
kiwi = Kiwi()
cnt = Counter()
for t in tr["text"]:
    toks = kiwi.tokenize(t)
    for a, b in zip(toks, toks[1:]):          # 붙어 있는 명사+명사 = 복합 현장 용어 후보
        if a.tag.startswith("NN") and b.tag.startswith("NN") and a.end == b.start:
            cnt[a.form + b.form] += 1
cnt.most_common(30)

In [ ]:
terms = pd.DataFrame([
    # [JOB, "용어", "설명"],
], columns=["job_key", "term", "description"])
old = pd.read_csv(DATA / "curated/field_terms.csv", dtype=str)
pd.concat([old, terms]).drop_duplicates(["job_key", "term"]).to_csv(
    DATA / "curated/field_terms.csv", index=False, encoding="utf-8-sig")

## 2) 기준 답: 템플릿을 복사해 작성 (엑셀에서 편집해도 됨, UTF-8 CSV 로 저장)

In [ ]:
gold_path = DATA / f"gold/{JOB}.csv"
if not gold_path.exists():
    pd.read_csv(DATA / "gold/_template.csv").to_csv(gold_path, index=False, encoding="utf-8-sig")
pd.read_csv(gold_path)

In [ ]:
from app.data_check import check_all
check_all(DATA) or "통과"